# Energy AI Hackathon 2026 — Full Pipeline\n## Train: 2024 | Test & Optimize: 2025\n### Load Forecasting (LightGBM) + MPC Battery Dispatch (CVXPY)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import time
import holidays

from sklearn.metrics import mean_squared_error, mean_absolute_error
import lightgbm as lgb
import cvxpy as cp

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## CONFIGURATION

In [ ]:
DATA_PATH_24 = "dataset_2024.csv"
DATA_PATH_25 = "dataset_2025.csv"

DELTA_T                  = 0.25       # hours per timestep
BATTERY_CAPACITY_KWH     = 16.0
BATTERY_MAX_POWER_KW     = 8.0
GRID_LIMIT_KW            = 6.0
BATTERY_ROUNDTRIP_EFF    = 0.90
BATTERY_EFFICIENCY       = np.sqrt(BATTERY_ROUNDTRIP_EFF)   # ~0.9487 per direction
INITIAL_SOC              = 0.50

# ── Load datasets ──────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_PATH_24)
test_df  = pd.read_csv(DATA_PATH_25)

for df in [train_df, test_df]:
    df["timestamp"] = pd.to_datetime(df["timestamp"])

print("Train shape:", train_df.shape, "| columns:", list(train_df.columns))
print("Test  shape:", test_df.shape,  "| date range:", test_df["timestamp"].min(), "→", test_df["timestamp"].max())
print("\nTrain head:")
train_df.head(3)

## BLOCK 1 — PREPROCESSING

In [ ]:
def preprocess_data(df):
    df = df.copy()
    df = df.replace("-", np.nan)
    numeric_cols = ["pv_kw", "p_battery_kw", "load_kw", "buy_price", "sell_price"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df[numeric_cols] = df[numeric_cols].fillna(0)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    # Enforce physical bounds
    df["load_kw"] = df["load_kw"].clip(lower=0)
    df["pv_kw"]   = df["pv_kw"].clip(lower=0)
    return df

train_df = preprocess_data(train_df)
test_df  = preprocess_data(test_df)

print("Missing after preprocessing — train:", train_df.isnull().sum().sum(), "| test:", test_df.isnull().sum().sum())
print(train_df.describe().round(3))

## BLOCK 2 — EDA (Exploratory Data Analysis)

In [ ]:
### EDA 2a — Full-year time series overview (2024 training data)

fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)

# Daily aggregation for readability
daily = train_df.set_index("timestamp").resample("D").mean()

axes[0].plot(daily.index, daily["load_kw"], color="steelblue", lw=1.2, label="Avg daily load (kW)")
axes[0].fill_between(daily.index, daily["load_kw"], alpha=0.2, color="steelblue")
axes[0].set_ylabel("Load (kW)")
axes[0].set_title("2024 Daily Average Load")
axes[0].legend()

axes[1].plot(daily.index, daily["pv_kw"], color="orange", lw=1.2, label="Avg daily PV (kW)")
axes[1].fill_between(daily.index, daily["pv_kw"], alpha=0.2, color="orange")
axes[1].set_ylabel("PV (kW)")
axes[1].set_title("2024 Daily Average PV Generation")
axes[1].legend()

axes[2].plot(daily.index, daily["buy_price"], color="red", lw=1, label="Buy price")
axes[2].plot(daily.index, daily["sell_price"], color="green", lw=1, label="Sell price")
axes[2].set_ylabel("Price (€/kWh)")
axes[2].set_title("Buy vs Sell Price (daily avg)")
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Annual load:  {train_df['load_kw'].sum() * DELTA_T:.1f} kWh")
print(f"Annual PV:    {train_df['pv_kw'].sum() * DELTA_T:.1f} kWh")
print(f"Self-sufficiency ratio: {(train_df['pv_kw'].clip(upper=train_df['load_kw']).sum() / train_df['load_kw'].sum() * 100):.1f}%")

In [ ]:
### EDA 2b — Daily load profile by season & day type

train_df["hour"]        = train_df["timestamp"].dt.hour
train_df["minute"]      = train_df["timestamp"].dt.minute
train_df["day_of_week"] = train_df["timestamp"].dt.dayofweek
train_df["month"]       = train_df["timestamp"].dt.month
train_df["slot"]        = train_df["hour"] * 4 + train_df["minute"] // 15  # 0-95

def season(m):
    if m in [12, 1, 2]:  return "Winter"
    if m in [3, 4, 5]:   return "Spring"
    if m in [6, 7, 8]:   return "Summer"
    return "Autumn"

train_df["season"]    = train_df["month"].map(season)
train_df["day_type"]  = train_df["day_of_week"].map(lambda d: "Weekend" if d >= 5 else "Weekday")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col, cmap, title in zip(
    axes,
    ["load_kw", "pv_kw"],
    [["#1f77b4","#ff7f0e","#2ca02c","#d62728"], ["#ff7f0e","#e6ac00","#ffd700","#c8860a"]],
    ["Load by Season (avg 15-min profile)", "PV by Season (avg 15-min profile)"]
):
    for i, s in enumerate(["Winter", "Spring", "Summer", "Autumn"]):
        grp = train_df[train_df["season"] == s].groupby("slot")[col].mean()
        ax.plot(grp.index / 4, grp.values, label=s, color=cmap[i], lw=1.8)
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("kW")
    ax.set_title(title)
    ax.set_xticks(range(0, 25, 3))
    ax.legend()

plt.tight_layout()
plt.show()

# Day-of-week heatmap for load
fig, ax = plt.subplots(figsize=(14, 4))
pivot = train_df.groupby(["day_of_week", "slot"])["load_kw"].mean().unstack()
im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd", origin="lower")
ax.set_yticks(range(7))
ax.set_yticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
ax.set_xticks(range(0, 96, 8))
ax.set_xticklabels([f"{h}:00" for h in range(0, 24, 2)], rotation=45)
ax.set_title("Load Heatmap: Day-of-Week × Time-of-Day (kW)")
plt.colorbar(im, ax=ax, label="kW")
plt.tight_layout()
plt.show()

In [ ]:
### EDA 2c — SOC reconstruction + corruption detection (2025)

def reconstruct_soc(df, initial_soc=INITIAL_SOC):
    df = df.copy()
    soc = initial_soc
    soc_values = []
    for p_battery in df["p_battery_kw"]:
        energy_change = p_battery * DELTA_T
        if p_battery < 0:   # charging
            soc += abs(energy_change) * BATTERY_EFFICIENCY / BATTERY_CAPACITY_KWH
        else:               # discharging
            soc -= energy_change / BATTERY_EFFICIENCY / BATTERY_CAPACITY_KWH
        soc = np.clip(soc, 0.0, 1.0)
        soc_values.append(soc)
    df["soc"] = soc_values
    return df

test_df_soc = reconstruct_soc(test_df)

# Energy balance residual: should be ~0 if data is clean
test_df_soc["grid_computed"] = (
    test_df_soc["load_kw"] - test_df_soc["pv_kw"] - test_df_soc["p_battery_kw"]
)
# Flag timesteps where |computed grid| > 6 kW (grid limit violation)
test_df_soc["balance_violation"] = (np.abs(test_df_soc["grid_computed"]) > GRID_LIMIT_KW + 0.1).astype(int)

daily_violations = test_df_soc.set_index("timestamp")["balance_violation"].resample("D").sum()
corruption_days = daily_violations[daily_violations > 0]

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)
axes[0].plot(test_df_soc["timestamp"], test_df_soc["soc"], lw=0.7, color="purple", label="SoC")
axes[0].axhspan(0, 0, alpha=0)
axes[0].set_ylabel("State of Charge")
axes[0].set_title("2025 Battery SoC (reconstructed from p_battery_kw)")
axes[0].legend()

axes[1].bar(corruption_days.index, corruption_days.values, color="red", width=1, label="Balance violations/day")
axes[1].set_ylabel("Violations per day")
axes[1].set_title("Energy Balance Violations (|grid| > 6 kW) → identifies corrupted battery data window")
axes[1].legend()

plt.tight_layout()
plt.show()

n_corrupted = test_df_soc["balance_violation"].sum()
print(f"Corrupted timesteps: {n_corrupted} ({n_corrupted/len(test_df_soc)*100:.1f}%)")
if len(corruption_days) > 0:
    print(f"Corruption window:   {corruption_days.index[0].date()} → {corruption_days.index[-1].date()}")

In [ ]:
### EDA 2d — Autocorrelation & seasonality decomposition

from pandas.plotting import autocorrelation_plot

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# ACF of load at lags 1..200 (covering 2 daily cycles = 192 slots)
load_series = train_df["load_kw"].values
lags = range(1, 201)
acf_vals = [pd.Series(load_series).autocorr(lag=l) for l in lags]
axes[0].bar(lags, acf_vals, width=1, color="steelblue", alpha=0.7)
axes[0].axhline(0, color="black", lw=0.8)
axes[0].axvline(96, color="red", lw=1.5, ls="--", label="1 day (96 steps)")
axes[0].axvline(192, color="orange", lw=1.5, ls="--", label="2 days")
axes[0].set_xlabel("Lag (15-min steps)")
axes[0].set_ylabel("Autocorrelation")
axes[0].set_title("Load Autocorrelation — daily seasonality visible")
axes[0].legend()

# Monthly distribution
monthly_load = train_df.groupby("month")["load_kw"].mean()
monthly_pv   = train_df.groupby("month")["pv_kw"].mean()
x = np.arange(1, 13)
axes[1].bar(x - 0.2, monthly_load, 0.4, label="Load", color="steelblue")
axes[1].bar(x + 0.2, monthly_pv,   0.4, label="PV", color="orange")
axes[1].set_xticks(x)
axes[1].set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
axes[1].set_ylabel("Average kW")
axes[1].set_title("Monthly Average Load vs PV")
axes[1].legend()

plt.tight_layout()
plt.show()

# Net-load distribution
train_df["net_load"] = train_df["load_kw"] - train_df["pv_kw"]
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(train_df["net_load"], bins=80, color="steelblue", edgecolor="white", alpha=0.8)
ax.axvline(0, color="red", lw=1.5, ls="--", label="Net load = 0 (PV covers load)")
ax.set_xlabel("Net Load (kW)")
ax.set_ylabel("Count")
ax.set_title("Net Load Distribution (load − PV) — negative = PV surplus")
ax.legend()
plt.tight_layout()
plt.show()
print(f"PV surplus timesteps: {(train_df['net_load'] < 0).sum()} / {len(train_df)} ({(train_df['net_load'] < 0).mean()*100:.1f}%)")

## BLOCK 3 — FEATURE ENGINEERING

In [ ]:
def create_features(df):
    df = df.copy()
    ts = df["timestamp"]

    # ── Calendar features ─────────────────────────────────────────────────────
    df["hour"]           = ts.dt.hour
    df["minute"]         = ts.dt.minute
    df["slot"]           = df["hour"] * 4 + df["minute"] // 15   # 0-95
    df["day_of_week"]    = ts.dt.dayofweek
    df["day_of_year"]    = ts.dt.dayofyear
    df["week_of_year"]   = ts.dt.isocalendar().week.astype(int)
    df["month"]          = ts.dt.month
    df["quarter"]        = ts.dt.quarter
    df["is_weekend"]     = (df["day_of_week"] >= 5).astype(int)

    # Italian holidays
    italy_hols = holidays.Italy(years=ts.dt.year.unique().tolist())
    df["is_holiday"] = ts.dt.date.isin(italy_hols).astype(int)

    # ── Cyclic encoding (preserves periodicity) ───────────────────────────────
    df["hour_sin"]  = np.sin(2 * np.pi * df["slot"] / 96)
    df["hour_cos"]  = np.cos(2 * np.pi * df["slot"] / 96)
    df["dow_sin"]   = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"]   = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
    df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)
    df["doy_sin"]   = np.sin(2 * np.pi * df["day_of_year"] / 365)
    df["doy_cos"]   = np.cos(2 * np.pi * df["day_of_year"] / 365)

    # ── Lag features (causal — only past values) ──────────────────────────────
    for lag in [1, 2, 4, 8, 12, 96, 192, 96*7]:   # 15min,30min,1h,2h,3h,1d,2d,1wk
        df[f"load_lag_{lag}"] = df["load_kw"].shift(lag)

    # ── Rolling statistics ────────────────────────────────────────────────────
    for window in [4, 12, 96]:    # 1h, 3h, 24h
        df[f"load_roll_mean_{window}"] = df["load_kw"].shift(1).rolling(window).mean()
        df[f"load_roll_std_{window}"]  = df["load_kw"].shift(1).rolling(window).std()

    # ── Same slot yesterday and last week (very strong features) ─────────────
    df["load_same_slot_yesterday"] = df["load_kw"].shift(96)
    df["load_same_slot_lastweek"]  = df["load_kw"].shift(96 * 7)

    # ── Tariff band feature (1=F1, 2=F2, 3=F3) ───────────────────────────────
    def tariff_band(row):
        h, d, hol = row["hour"], row["day_of_week"], row["is_holiday"]
        if hol or d == 6:   return 3
        if d <= 4 and 8 <= h < 19: return 1
        if (d <= 4 and (7 <= h < 8 or 19 <= h < 23)) or (d == 5 and 7 <= h < 23): return 2
        return 3
    df["tariff_band"] = df.apply(tariff_band, axis=1)

    df = df.bfill()
    return df

train_feat = create_features(train_df)
test_feat  = create_features(test_df)

FEATURE_COLS = [c for c in train_feat.columns if c not in
    ["timestamp", "load_kw", "pv_kw", "p_battery_kw", "buy_price", "sell_price",
     "grid_kw", "soc", "net_load", "season", "day_type", "minute"]]

print(f"Feature columns ({len(FEATURE_COLS)}):", FEATURE_COLS)

## BLOCK 4 — TRAIN / VALIDATION SPLIT (2024 only, no leakage)

In [ ]:
# Temporal split — last 20% of 2024 as validation (≈ Nov-Dec 2024)
split_idx   = int(len(train_feat) * 0.80)
tr_split    = train_feat.iloc[:split_idx].copy()
val_split   = train_feat.iloc[split_idx:].copy()

X_tr  = tr_split[FEATURE_COLS]
y_tr  = tr_split["load_kw"]
X_val = val_split[FEATURE_COLS]
y_val = val_split["load_kw"]

print(f"Train slice : {tr_split['timestamp'].min().date()} → {tr_split['timestamp'].max().date()}  ({len(tr_split)} rows)")
print(f"Val slice   : {val_split['timestamp'].min().date()} → {val_split['timestamp'].max().date()}  ({len(val_split)} rows)")

## BLOCK 5 — FORECASTING MODEL (LightGBM)

In [ ]:
def compute_metrics(actual, predicted, label=""):
    rmse  = np.sqrt(mean_squared_error(actual, predicted))
    mae   = mean_absolute_error(actual, predicted)
    nrmse = rmse / np.mean(actual) * 100
    if label:
        print(f"  [{label}]  RMSE={rmse:.4f} kW  |  MAE={mae:.4f} kW  |  NRMSE={nrmse:.2f}%")
    return {"RMSE": rmse, "MAE": mae, "NRMSE": nrmse}

# ── LightGBM — tuned for time-series residential load ─────────────────────────
lgb_params = {
    "objective":        "regression",
    "metric":           "rmse",
    "boosting_type":    "gbdt",
    "num_leaves":       127,
    "max_depth":        -1,
    "learning_rate":    0.05,
    "n_estimators":     2000,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "min_child_samples": 20,
    "reg_alpha":        0.1,
    "reg_lambda":       0.1,
    "n_jobs":           -1,
    "random_state":     42,
    "verbose":          -1,
}

lgb_model = lgb.LGBMRegressor(**lgb_params)
lgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)]
)

val_pred_lgb = lgb_model.predict(X_val)
val_pred_lgb = np.clip(val_pred_lgb, 0, None)

print("Validation metrics (2024 held-out):")
val_metrics = compute_metrics(y_val, val_pred_lgb, "LightGBM val")

# ── Feature importance ────────────────────────────────────────────────────────
importance = pd.Series(lgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
importance.head(20).plot.bar(ax=ax, color="steelblue")
ax.set_title("Top 20 Feature Importances — LightGBM")
ax.set_ylabel("Importance (gain)")
plt.tight_layout()
plt.show()

## BLOCK 6 — FORECAST 2025 TEST SET (no retraining)

In [ ]:
# The lag/rolling features in test_feat were built from 2025 data only.
# For proper causal forecasting we append the tail of 2024 before building features.
tail_2024 = train_df.tail(96 * 8).copy()   # 8 days of lookback
combined  = pd.concat([tail_2024, test_df], ignore_index=True)
combined_feat = create_features(combined)
test_feat = combined_feat.iloc[len(tail_2024):].copy().reset_index(drop=True)

X_test = test_feat[FEATURE_COLS]
test_pred = lgb_model.predict(X_test)
test_pred = np.clip(test_pred, 0, None)

test_feat["forecast_load_kw"] = test_pred

print("2025 TEST SET metrics:")
test_metrics = compute_metrics(test_feat["load_kw"], test_pred, "LightGBM 2025")

# ── Forecast vs actual — sample 2 weeks ──────────────────────────────────────
sample = test_feat[(test_feat["timestamp"] >= "2025-06-01") & (test_feat["timestamp"] < "2025-06-15")]
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(sample["timestamp"], sample["load_kw"],        lw=1.2, label="Actual",   color="steelblue")
ax.plot(sample["timestamp"], sample["forecast_load_kw"], lw=1.2, label="Forecast", color="orange", alpha=0.85)
ax.set_title("2025 Load: Actual vs LightGBM Forecast (2 weeks sample)")
ax.set_ylabel("kW")
ax.legend()
plt.tight_layout()
plt.show()

## BLOCK 7 — BASELINES (2025)

In [ ]:
def compute_bill(df, p_battery_col="p_battery_kw"):
    """Compute electricity bill from a dispatch schedule."""
    df = df.copy()
    p_bat = df[p_battery_col].values
    load  = df["load_kw"].values
    pv    = df["pv_kw"].values
    buy   = df["buy_price"].values
    sell  = df["sell_price"].values

    p_grid = load - pv - p_bat
    p_grid = np.clip(p_grid, -GRID_LIMIT_KW, GRID_LIMIT_KW)

    import_kw  = np.maximum(p_grid,  0)
    export_kw  = np.maximum(-p_grid, 0)
    cost        = import_kw * buy  * DELTA_T
    revenue     = export_kw * sell * DELTA_T
    net_cost    = cost - revenue
    return net_cost.sum()

# ── Baseline A: actual 2025 on-site controller ────────────────────────────────
bill_A = compute_bill(test_df, "p_battery_kw")
print(f"Baseline A (historical controller): €{bill_A:.2f}")

# ── Baseline B: zero-intelligence (battery disabled, PV → load first) ─────────
test_B = test_df.copy()
test_B["p_battery_kw"] = 0.0
bill_B = compute_bill(test_B, "p_battery_kw")
print(f"Baseline B (zero-intelligence):     €{bill_B:.2f}")

## BLOCK 8 — MPC ROLLING-HORIZON OPTIMIZER (CVXPY)

In [ ]:
def rolling_horizon_optimizer(df, forecast_load, horizon=96, verbose=False):
    """
    MPC battery dispatch controller.

    Parameters
    ----------
    df            : full 2025 DataFrame with actual pv_kw, buy_price, sell_price
    forecast_load : array of forecasted load (same length as df)
    horizon       : look-ahead window in timesteps (default 96 = 24 h)
    """
    n = len(df)
    pv    = df["pv_kw"].values
    buy   = df["buy_price"].values
    sell  = df["sell_price"].values

    opt_battery = np.zeros(n)
    soc_trace   = np.zeros(n)
    soc_t       = INITIAL_SOC

    t0 = time.time()
    for t in range(n):
        H_end = min(t + horizon, n)
        H     = H_end - t

        load_h = forecast_load[t:H_end]
        pv_h   = pv[t:H_end]
        buy_h  = buy[t:H_end]
        sell_h = sell[t:H_end]

        # ── CVXPY variables ──────────────────────────────────────────────────
        p_bat  = cp.Variable(H)    # battery power (+ = discharge, - = charge)
        p_grid = cp.Variable(H)    # grid power    (+ = import,    - = export)
        soc    = cp.Variable(H + 1)

        # ── Objective: minimise net bill over horizon ──────────────────────
        import_power = cp.pos(p_grid)
        export_power = cp.pos(-p_grid)
        cost         = cp.sum(cp.multiply(import_power, buy_h)  * DELTA_T)
        revenue      = cp.sum(cp.multiply(export_power, sell_h) * DELTA_T)
        objective    = cp.Minimize(cost - revenue)

        # ── Constraints ────────────────────────────────────────────────────
        constraints = [
            # Energy balance
            p_grid == load_h - pv_h - p_bat,
            # Grid limits
            p_grid >= -GRID_LIMIT_KW,
            p_grid <=  GRID_LIMIT_KW,
            # Battery power limits
            p_bat >= -BATTERY_MAX_POWER_KW,
            p_bat <=  BATTERY_MAX_POWER_KW,
            # SoC dynamics
            soc[0] == soc_t,
            soc[1:] >= 0.0,
            soc[1:] <= 1.0,
        ]

        # SoC update (linearized split-efficiency)
        for k in range(H):
            # Charging (p_bat < 0): SoC increases; Discharging (p_bat > 0): SoC decreases
            # Linear approximation: use average efficiency both ways
            constraints.append(
                soc[k + 1] == soc[k]
                - p_bat[k] * DELTA_T / (BATTERY_EFFICIENCY * BATTERY_CAPACITY_KWH)
            )

        prob = cp.Problem(objective, constraints)
        try:
            prob.solve(solver=cp.CLARABEL, warm_start=True)
        except Exception:
            prob.solve(solver=cp.SCS, warm_start=True)

        if prob.status in ["optimal", "optimal_inaccurate"] and p_bat.value is not None:
            p_bat_now = float(p_bat.value[0])
        else:
            p_bat_now = 0.0

        p_bat_now = np.clip(p_bat_now, -BATTERY_MAX_POWER_KW, BATTERY_MAX_POWER_KW)

        # Apply SoC update with proper split efficiency
        if p_bat_now < 0:   # charging
            soc_t += abs(p_bat_now) * DELTA_T * BATTERY_EFFICIENCY / BATTERY_CAPACITY_KWH
        else:                # discharging
            soc_t -= p_bat_now * DELTA_T / BATTERY_EFFICIENCY / BATTERY_CAPACITY_KWH
        soc_t = np.clip(soc_t, 0.0, 1.0)

        opt_battery[t]  = p_bat_now
        soc_trace[t]    = soc_t

        if verbose and t % 1000 == 0:
            elapsed = time.time() - t0
            print(f"  t={t}/{n}  SoC={soc_t:.3f}  elapsed={elapsed:.1f}s")

    elapsed = time.time() - t0
    result_df = df.copy()
    result_df["optimized_battery_kw"] = opt_battery
    result_df["optimized_soc"]        = soc_trace
    result_df["optimized_grid_kw"]    = np.clip(
        result_df["load_kw"] - result_df["pv_kw"] - opt_battery,
        -GRID_LIMIT_KW, GRID_LIMIT_KW
    )
    return result_df, elapsed

print("MPC optimizer defined. Run next cell to execute on 2025 data (may take several minutes).")

## BLOCK 9 — RUN MPC (H=96, forecast-based)

In [ ]:
HORIZON = 96    # 24-hour look-ahead

print(f"Running MPC with H={HORIZON} (24 h horizon) on 2025 data...")
print("Tip: this runs one LP per timestep — ~35 000 solves. Expect 5-15 min on a standard laptop.")

forecast_arr = test_feat["forecast_load_kw"].values
mpc_df, mpc_time = rolling_horizon_optimizer(test_df, forecast_arr, horizon=HORIZON, verbose=True)

# Attach forecast column so the dispatch plot can show it
mpc_df["forecast_load_kw"] = forecast_arr

bill_mpc = compute_bill(mpc_df, "optimized_battery_kw")

print(f"\nMPC elapsed: {mpc_time:.1f}s")
print(f"MPC bill (forecast-based, H={HORIZON}): €{bill_mpc:.2f}")
print(f"Savings vs Baseline A: €{bill_A - bill_mpc:.2f}  ({(bill_A - bill_mpc)/bill_A*100:.1f}%)")
print(f"Savings vs Baseline B: €{bill_B - bill_mpc:.2f}  ({(bill_B - bill_mpc)/bill_B*100:.1f}%)")

## BLOCK 10 — ORACLE GAP (perfect forecast)

In [ ]:
print("Running Oracle MPC (actual 2025 load as perfect forecast)...")
oracle_arr = test_df["load_kw"].values
oracle_df, oracle_time = rolling_horizon_optimizer(test_df, oracle_arr, horizon=HORIZON, verbose=False)

bill_oracle = compute_bill(oracle_df, "optimized_battery_kw")

oracle_savings   = bill_A      - bill_oracle
forecast_savings = bill_A      - bill_mpc
oracle_gap_eur   = bill_mpc    - bill_oracle
oracle_gap_pct   = oracle_gap_eur / bill_oracle * 100 if bill_oracle > 0 else 0

print(f"\nOracle bill       : €{bill_oracle:.2f}")
print(f"Oracle savings vs A: €{oracle_savings:.2f}  ({oracle_savings/bill_A*100:.1f}%)")
print(f"Forecast savings vs A: €{forecast_savings:.2f}  ({forecast_savings/bill_A*100:.1f}%)")
print(f"Oracle gap        : €{oracle_gap_eur:.2f}  ({oracle_gap_pct:.1f}%) — cost of forecast error")

## BLOCK 11 — EXTENSION 1: Horizon Sensitivity Analysis

In [ ]:
HORIZONS = [4, 24, 96]   # 1h, 6h, 24h  (add more if time allows)

horizon_results = []
for H in HORIZONS:
    print(f"  Horizon H={H} ({H*0.25:.0f}h)...")
    h_df, h_time = rolling_horizon_optimizer(test_df, forecast_arr, horizon=H, verbose=False)
    h_bill = compute_bill(h_df, "optimized_battery_kw")
    savings_vs_A = bill_A - h_bill
    horizon_results.append({
        "Horizon (steps)": H,
        "Horizon (hours)": H * 0.25,
        "Bill (€)":        round(h_bill, 2),
        "Savings vs A (€)":round(savings_vs_A, 2),
        "Savings vs A (%)":round(savings_vs_A / bill_A * 100, 2),
        "Time (s)":        round(h_time, 1),
    })

horizon_df = pd.DataFrame(horizon_results)
print("\nHorizon Sensitivity — Extension 1:")
print(horizon_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(horizon_df["Horizon (hours)"], horizon_df["Savings vs A (€)"], "o-", color="steelblue", lw=2, ms=8)
axes[0].set_xlabel("Horizon (hours)")
axes[0].set_ylabel("Savings vs Baseline A (€)")
axes[0].set_title("Savings vs Look-Ahead Horizon")

axes[1].plot(horizon_df["Horizon (hours)"], horizon_df["Time (s)"], "o-", color="orange", lw=2, ms=8)
axes[1].set_xlabel("Horizon (hours)")
axes[1].set_ylabel("Computation Time (s)")
axes[1].set_title("Compute Time vs Horizon")

plt.tight_layout()
plt.show()

print("\nRecommendation: H=96 (24h) — captures full overnight charge/discharge cycle.")

## BLOCK 12 — MANDATORY: March Week 3 Dispatch Plot

In [ ]:
# March Week 3: Mon 17 Mar – Sun 23 Mar 2025
week_mask = (mpc_df["timestamp"] >= "2025-03-17") & (mpc_df["timestamp"] < "2025-03-24")
wk = mpc_df[week_mask].copy()
wk["soc_reconstructed"] = reconstruct_soc(wk)["soc"]

fig, axes = plt.subplots(5, 1, figsize=(18, 14), sharex=True)
fig.suptitle("March Week 3 (17–23 Mar 2025) — MPC Dispatch", fontsize=14, fontweight="bold")

axes[0].plot(wk["timestamp"], wk["load_kw"],  lw=1.4, color="steelblue", label="Load")
axes[0].plot(wk["timestamp"], wk["forecast_load_kw"] if "forecast_load_kw" in wk else wk["load_kw"],
             lw=1, color="orange", ls="--", label="Forecast", alpha=0.8)
axes[0].set_ylabel("kW")
axes[0].set_title("Load (actual vs forecast)")
axes[0].legend(fontsize=9)

axes[1].fill_between(wk["timestamp"], wk["pv_kw"], alpha=0.5, color="gold")
axes[1].plot(wk["timestamp"], wk["pv_kw"], lw=1.2, color="darkorange", label="PV")
axes[1].set_ylabel("kW")
axes[1].set_title("PV Generation")
axes[1].legend(fontsize=9)

p_bat = wk["optimized_battery_kw"]
axes[2].fill_between(wk["timestamp"], p_bat.clip(upper=0), alpha=0.5, color="green", label="Charging (−)")
axes[2].fill_between(wk["timestamp"], p_bat.clip(lower=0), alpha=0.5, color="red",   label="Discharging (+)")
axes[2].plot(wk["timestamp"], p_bat, lw=1, color="black")
axes[2].axhline(0, color="grey", lw=0.8, ls="--")
axes[2].set_ylabel("kW")
axes[2].set_title("Battery Power (−=charge, +=discharge)")
axes[2].legend(fontsize=9)

p_grid = wk["optimized_grid_kw"]
axes[3].fill_between(wk["timestamp"], p_grid.clip(lower=0),  alpha=0.5, color="tomato",      label="Import (+)")
axes[3].fill_between(wk["timestamp"], p_grid.clip(upper=0),  alpha=0.5, color="mediumseagreen", label="Export (−)")
axes[3].plot(wk["timestamp"], p_grid, lw=1, color="black")
axes[3].axhline(0, color="grey", lw=0.8, ls="--")
axes[3].set_ylabel("kW")
axes[3].set_title("Grid Power (+=import, −=export)")
axes[3].legend(fontsize=9)

axes[4].plot(wk["timestamp"], wk["optimized_soc"] * 100, lw=1.8, color="purple", label="SoC")
axes[4].fill_between(wk["timestamp"], wk["optimized_soc"] * 100, alpha=0.2, color="purple")
axes[4].axhline(20, color="red",  lw=0.8, ls="--", label="SoC min 20%")
axes[4].axhline(90, color="blue", lw=0.8, ls="--", label="SoC max 90%")
axes[4].set_ylabel("%")
axes[4].set_ylim(-5, 105)
axes[4].set_title("Battery State of Charge")
axes[4].legend(fontsize=9)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d\n%H:%M"))
    ax.xaxis.set_major_locator(mdates.DayLocator())

plt.tight_layout()
plt.savefig("march_week3_dispatch.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: march_week3_dispatch.png")

## BLOCK 13 — FINAL RESULTS TABLE

In [ ]:
print("=" * 60)
print("FORECASTING RESULTS — 2025 TEST SET")
print("=" * 60)
print(f"  RMSE  : {test_metrics['RMSE']:.4f} kW")
print(f"  MAE   : {test_metrics['MAE']:.4f} kW")
print(f"  NRMSE : {test_metrics['NRMSE']:.2f}%  ← primary ranking metric")

print("\n" + "=" * 60)
print("ELECTRICITY BILL COMPARISON — 2025")
print("=" * 60)

scenarios = [
    ("Baseline A (historical)", bill_A,      0,               0),
    ("Baseline B (zero-intel)", bill_B,      bill_A - bill_B, (bill_A-bill_B)/bill_A*100),
    ("MPC H=96 (forecast)",     bill_mpc,    bill_A - bill_mpc, (bill_A-bill_mpc)/bill_A*100),
    ("Oracle H=96 (perfect)",   bill_oracle, bill_A - bill_oracle, (bill_A-bill_oracle)/bill_A*100),
]

print(f"\n{'Scenario':<28} {'Bill (€)':>10} {'Savings vs A (€)':>18} {'Savings vs A (%)':>17}")
print("-" * 75)
for name, bill, sav_eur, sav_pct in scenarios:
    print(f"{name:<28} {bill:>10.2f} {sav_eur:>18.2f} {sav_pct:>16.1f}%")

print("\n" + "=" * 60)
print("ORACLE GAP")
print("=" * 60)
print(f"  Forecast savings : €{forecast_savings:.2f} ({forecast_savings/bill_A*100:.1f}%)")
print(f"  Oracle savings   : €{oracle_savings:.2f} ({oracle_savings/bill_A*100:.1f}%)")
print(f"  Oracle gap       : €{oracle_gap_eur:.2f} ({oracle_gap_pct:.1f}%) — cost of forecast imperfection")

print("\n" + "=" * 60)
print("EXTENSION 1 — HORIZON SENSITIVITY (H=4, 24, 96 steps)")
print("=" * 60)
print(horizon_df[["Horizon (hours)","Bill (€)","Savings vs A (€)","Savings vs A (%)","Time (s)"]].to_string(index=False))

## BLOCK 14 — DAY 2 SURPRISE DATASET (run without retraining)
# When the Day 2 dataset is released, drop it in as dataset_day2.csv and run this cell.

In [ ]:
DAY2_PATH = "dataset_day2.csv"   # update path when released

try:
    day2_raw = pd.read_csv(DAY2_PATH)
    day2_raw = preprocess_data(day2_raw)
    day2_feat = create_features(day2_raw)
    X_day2 = day2_feat[FEATURE_COLS]
    day2_pred = lgb_model.predict(X_day2)
    day2_pred = np.clip(day2_pred, 0, None)
    day2_metrics = compute_metrics(day2_feat["load_kw"], day2_pred, "LightGBM Day2")
    print(f"\nDay 2 NRMSE: {day2_metrics['NRMSE']:.2f}%  vs 2025 NRMSE: {test_metrics['NRMSE']:.2f}%")
except FileNotFoundError:
    print(f"Day 2 dataset not found at '{DAY2_PATH}' — place file there when released and re-run.")

**RUN MPC OPTIMIZER SKELETON**

**RESULTS COMPARISON**